# Protein Filtering

---

## Setup

### Load modules

In [ ]:
# ── Standard libraries ──
import os, sys, copy, random, multiprocessing
import numpy as np
import pandas as pd
import yaml
import seaborn as sns
import matplotlib.pyplot as plt

# ── Mosaic ──
sys.path = [p for p in sys.path if p]
import missionbio.mosaic as ms

# ── Plotly ──
import plotly.graph_objects as go
import plotly.offline as pyo
pyo.init_notebook_mode()

# ── Interactive tables ──
from itables import init_notebook_mode
import itables.options as opt
init_notebook_mode(all_interactive=True)
opt.lengthMenu = [5, 10, 20, 50, 100, 200, 500]
opt.maxBytes = 0
opt.maxColumns = 0

%matplotlib inline

### Reproducibility

In [ ]:
%env PYTHONHASHSEED=42
random.seed(42)
np.random.seed(42)

## Experiment configuration

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SET EXPERIMENT NAME — everything else is read from the config
# ═══════════════════════════════════════════════════════════════════

EXPERIMENT = "MDS008"

SKIP_CHECK = True

In [ ]:
# ── Project root ──
wd = os.path.dirname(os.path.dirname(os.getcwd()))

# ── Load experiment_config.yml ──
config_path = os.path.join(wd, "metadata", EXPERIMENT, "experiment_config.yml")
with open(config_path) as f:
    config = yaml.safe_load(f)

IS_MULTIPLEXED = config.get("multiplexed", False)
N_DONORS       = config.get("n_samples_multiplexed", 1)

print(f"Experiment:  {EXPERIMENT}")
print(f"Multiplexed: {IS_MULTIPLEXED} ({N_DONORS} donor(s))")

## Load per-patient H5 files

Auto-discovers all `*_protein_unfiltered.dna+protein.h5` files
from the preprocessing export step.

In [ ]:
# ── Find per-patient H5 files ──
if IS_MULTIPLEXED:
    h5_base = os.path.join(wd, "processed_data", "h5", EXPERIMENT, "demultiplexed")
else:
    h5_base = os.path.join(wd, "processed_data", "h5", EXPERIMENT)

samples = {}  # patient_id → sample object

for patient_id in sorted(os.listdir(h5_base)):
    patient_dir = os.path.join(h5_base, patient_id)
    if not os.path.isdir(patient_dir):
        continue

    # Find the protein_unfiltered H5 file
    h5_files = [f for f in os.listdir(patient_dir)
                if f.endswith(".h5") and "protein_unfiltered" in f.lower()]

    if not h5_files:
        # Fallback: any H5 file
        h5_files = [f for f in os.listdir(patient_dir) if f.endswith(".h5")]

    if not h5_files:
        print(f"  ⚠ {patient_id}: no H5 files found, skipping")
        continue

    h5_path = os.path.join(patient_dir, h5_files[0])
    sample = ms.load(h5_path, filter_variants=False, filter_cells=False)
    samples[patient_id] = sample
    print(f"  ✓ {patient_id}: {sample.dna.shape[0]} cells, "
          f"{sample.protein.shape[1] if sample.protein is not None else 0} protein features "
          f"← {h5_files[0]}")

print(f"\nLoaded {len(samples)} patient(s)")

### Data overview

In [ ]:
for patient_id, sample in samples.items():
    print(f"\n── {patient_id} ──")
    for name, assay in [("DNA", sample.dna), ("Protein", sample.protein)]:
        if assay is not None:
            print(f"  {name}: {assay}")
            print(f"    row_attrs: {list(assay.row_attrs.keys())}")
            print(f"    col_attrs: {list(assay.col_attrs.keys())}")
            print(f"    layers:    {list(assay.layers.keys())}")

## Protein filtering

Filter cells by total protein read counts using an IQR-based outlier gate.
Cells above Q75 + 3×IQR are removed (abnormally high protein capture = likely debris/aggregates).

In [ ]:
filtered_samples = {}

for patient_id, sample in samples.items():
    print(f"\n{'='*50}")
    print(f"{patient_id}")
    print(f"{'='*50}")

    if sample.protein is None:
        print("  ⚠ No protein assay, skipping")
        filtered_samples[patient_id] = sample
        continue

    # ── Total protein reads per cell ──
    protein_reads = sample.protein.get_attribute('read_counts', constraint='row+col')
    total_reads = protein_reads.sum(axis=1)
    total_reads_df = pd.DataFrame({
        "Total_reads_Protein_per_Cell": total_reads,
        "Sample": patient_id,
    })

    # ── Pre-filtering stats ──
    stats_pre = total_reads_df["Total_reads_Protein_per_Cell"].describe(
        percentiles=[.10, .25, .5, .75, .90]
    )
    print(f"  Pre-filter:  {len(total_reads_df)} cells")
    print(f"    median={stats_pre['50%']:.0f}, Q25={stats_pre['25%']:.0f}, Q75={stats_pre['75%']:.0f}")

    # ── Pre-filtering violin ──
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    sns.violinplot(data=total_reads_df, x="Sample", y="Total_reads_Protein_per_Cell", ax=axes[0])
    axes[0].set_title(f"{patient_id} — pre-filter")

    # ── IQR-based upper filter ──
    q25 = stats_pre['25%']
    q75 = stats_pre['75%']
    iqr = q75 - q25
    upper_threshold = q75 + 3 * iqr

    print(f"    IQR={iqr:.0f}, upper threshold={upper_threshold:.0f}")

    # ── Apply filter ──
    keep_mask = total_reads <= upper_threshold
    keep_barcodes = total_reads_df.index[keep_mask].tolist()
    filtered_sample = sample[keep_barcodes]

    n_removed = len(total_reads_df) - len(keep_barcodes)
    print(f"  Post-filter: {len(keep_barcodes)} cells (removed {n_removed})")

    # ── Post-filtering violin ──
    total_reads_post = filtered_sample.protein.get_attribute('read_counts', constraint='row+col').sum(axis=1)
    total_reads_post_df = pd.DataFrame({
        "Total_reads_Protein_per_Cell": total_reads_post,
        "Sample": patient_id,
    })
    sns.violinplot(data=total_reads_post_df, x="Sample", y="Total_reads_Protein_per_Cell", ax=axes[1])
    axes[1].set_title(f"{patient_id} — post-filter")

    plt.tight_layout()
    plt.show()

    filtered_samples[patient_id] = filtered_sample

⚠️ **CHECKPOINT** — inspect violin plots before continuing.

In [ ]:
if not SKIP_CHECK:
    response = input("Continue? (y/n): ").strip().lower()
    if response not in ('y', 'yes'):
        raise SystemExit("Stopped by user.")
    print("✓ Continuing")
else:
    print("⚠️  SKIP_CHECK=True — continuing")

## Export protein-filtered H5

Saves per-patient filtered H5 files alongside the unfiltered ones.

In [ ]:
for patient_id, sample in filtered_samples.items():
    # Output path — same directory as the input, with _Filtered suffix
    if IS_MULTIPLEXED:
        out_dir = os.path.join(wd, "processed_data", "h5", EXPERIMENT, "demultiplexed", patient_id)
    else:
        out_dir = os.path.join(wd, "processed_data", "h5", EXPERIMENT, patient_id)

    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{patient_id}_protein_filtered.dna+protein.h5")

    try:
        ms.save(sample, out_path)
        dna_n = sample.dna.shape[0]
        prot_n = sample.protein.shape[1] if sample.protein is not None else 0
        print(f"  ✓ {patient_id}: {dna_n} cells, {prot_n} protein features → {out_path}")
    except Exception as e:
        print(f"  ⚠ {patient_id}: save failed ({e}), retrying...")
        try:
            if os.path.exists(out_path):
                os.remove(out_path)
            ms.save(sample, out_path)
            print(f"  ✓ {patient_id}: saved on retry")
        except Exception as e2:
            print(f"  ✗ {patient_id}: failed — {e2}")

print("\nDone.")

## BAM demultiplexing

For multiplexed experiments, splits the experiment-level BAM file into
per-patient BAMs using `sinto filterbarcodes`. Requires `pysam` and `sinto`.

Skipped for single-donor experiments.

In [ ]:
import subprocess, shutil

if IS_MULTIPLEXED:
    import pysam

    # ── Locate multiplexed BAM ──
    bam_dir = os.path.join(wd, "raw_data", "bam", EXPERIMENT)
    # Check multiplexed subdir first, then experiment root
    mux_dir = os.path.join(bam_dir, "multiplexed")
    if os.path.isdir(mux_dir):
        bam_search = mux_dir
    else:
        bam_search = bam_dir

    bam_files = [f for f in os.listdir(bam_search) if f.endswith(".bam")]
    assert bam_files, f"No BAM files found in {bam_search}"
    BAM_PATH = os.path.join(bam_search, bam_files[0])
    print(f"Multiplexed BAM: {BAM_PATH}")

    # ── Index BAM if needed ──
    bai_path = BAM_PATH + ".bai"
    if not os.path.exists(bai_path):
        print("  Indexing BAM...")
        pysam.index(BAM_PATH)
        print("  ✓ Indexed")
    else:
        print("  ✓ Index exists")

    # ── Build barcode → patient TSV for sinto ──
    # Uses barcodes from the protein-filtered samples (post-QC)
    barcode_tsv_path = os.path.join(
        wd, "results", "demultiplexing", EXPERIMENT, "bam_demultiplexing_barcodes.tsv"
    )
    os.makedirs(os.path.dirname(barcode_tsv_path), exist_ok=True)

    rows = []
    for patient_id, sample in filtered_samples.items():
        barcodes = np.asarray(sample.dna.row_attrs["barcode"]).astype(str)
        for bc in barcodes:
            # Sinto expects barcode with -1 suffix
            bc_sinto = bc if bc.endswith("-1") else bc + "-1"
            rows.append(f"{bc_sinto}\t{patient_id}")

    with open(barcode_tsv_path, "w") as f:
        f.write("\n".join(rows))

    print(f"  Barcode TSV: {len(rows)} barcodes across {len(filtered_samples)} patients")
    print(f"  Written: {barcode_tsv_path}")

    # ── Run sinto filterbarcodes ──
    demux_bam_dir = os.path.join(wd, "raw_data", "bam", EXPERIMENT, "demultiplexed")
    os.makedirs(demux_bam_dir, exist_ok=True)
    n_cores = max(1, os.cpu_count() - 2)

    # Absolute path to sinto in its own conda env
    SINTO_BIN = "/project/meadlab/kgurashi/miniconda3/envs/sinto/bin/sinto"

    assert os.path.exists(SINTO_BIN), f"sinto binary not found at {SINTO_BIN}"

    sinto_cmd = [
        SINTO_BIN, "filterbarcodes",
        "-b", BAM_PATH,
        "-c", barcode_tsv_path,
        "-p", str(n_cores),
        "--outdir", demux_bam_dir,
        "--barcodetag", "RG",
    ]
    print(f"  Running: {' '.join(sinto_cmd)}")
    result = subprocess.run(sinto_cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(f"  ⚠ sinto failed:\n{result.stderr}")
    else:
        print("  ✓ sinto completed")

    # ── Move per-patient BAMs into patient subdirectories ──
    for patient_id in filtered_samples.keys():
        src_bam = os.path.join(demux_bam_dir, f"{patient_id}.bam")
        if not os.path.exists(src_bam):
            print(f"  ⚠ {patient_id}: BAM not found at {src_bam}")
            continue

        dst_dir = os.path.join(demux_bam_dir, patient_id)
        os.makedirs(dst_dir, exist_ok=True)
        dst_bam = os.path.join(dst_dir, f"{patient_id}.bam")

        shutil.move(src_bam, dst_bam)
        print(f"  ✓ {patient_id}: moved to {dst_bam}")

        # Index the per-patient BAM
        pysam.index(dst_bam)
        print(f"    ✓ Indexed")

    print("\nBAM demultiplexing complete.")

else:
    print("Single donor — BAM demultiplexing skipped")

## Create per-patient sample analysis notebooks

For each patient that passed protein filtering, copies the
`03_Sample_Analysis_Template.ipynb` from `scripts/templates/`
and sets both `EXPERIMENT` and `PATIENT_ID`. Existing notebooks
are never overwritten.

In [ ]:
# ── Create per-patient 03 notebooks from template ──
template_path = os.path.join(wd, "scripts", "templates", "03_Sample_Analysis_Template.ipynb")
nb_outdir = os.path.join(wd, "notebooks", "03_sample_analysis")
os.makedirs(nb_outdir, exist_ok=True)

if os.path.exists(template_path):
    import json as _json

    with open(template_path) as f:
        template_nb = _json.load(f)

    for patient_id in sorted(filtered_samples.keys()):
        nb_path = os.path.join(nb_outdir, f"03_Sample_Analysis_{EXPERIMENT}_{patient_id}.ipynb")

        if os.path.exists(nb_path):
            print(f"  ✓ {patient_id}: notebook exists — not overwritten")
            continue

        # Deep copy and replace EXPERIMENT + PATIENT_ID
        patient_nb = _json.loads(_json.dumps(template_nb))

        for cell in patient_nb["cells"]:
            if cell["cell_type"] != "code":
                continue
            new_source = []
            for line in cell["source"]:
                stripped = line.rstrip("\n")
                nl = "\n" if line.endswith("\n") else ""
                if stripped.startswith("EXPERIMENT = "):
                    new_source.append(f'EXPERIMENT = "{EXPERIMENT}"' + nl)
                elif stripped.startswith("PATIENT_ID = "):
                    new_source.append(f'PATIENT_ID = "{patient_id}"' + nl)
                else:
                    new_source.append(line)
            cell["source"] = new_source

        with open(nb_path, "w") as f:
            _json.dump(patient_nb, f, indent=1, ensure_ascii=False)

        print(f"  + {patient_id}: created {os.path.basename(nb_path)}")

    print(f"\nDone. Notebooks in: {nb_outdir}")
else:
    print(f"⚠ Template not found: {template_path}")
    print("  Place 03_Sample_Analysis_Template.ipynb in scripts/templates/")